# Teste de corregistro MNI → imagens ADNI (global rígido → afim + local SyN)

Móvel = MNI152 2009c; fixas = `N_POR_DIAG` imagens de cada diagnóstico em `DIAGS` (sorteio com `SEED`: 1 imagem por paciente, paciente não se repete entre diagnósticos; só imagens do `adnimerged.csv` com T1 e `regions` em disco).

Variantes (escolha em `RODAR`, célula 1):

- `sem_mask`: SyN padrão do ANTsPy, sem máscara fixa (= o que o `2.2` executou, já que `fixed_mask` é ignorado).
- `com_mask`: igual, mas com `mask` = máscara da imagem fixa.
- `syn_robusto`: sem máscara, SyN com os parâmetros do `3.1 --src oasis` (CC raio 4, `100x70x50x20`).

Saídas em `test_dvf/{DIAG}_{ID}/`, todas no grid da imagem fixa: `fixed`, `regions` (labels originais, sem transformação), `mni_warped_{variante}` e `warp_{variante}_1Warp`. No ITK-SNAP: main = `mni_warped_{variante}`, segmentação = `regions`. Registro retomável: variantes já salvas são puladas.

In [2]:
import os
import shutil

import ants
import numpy as np
import pandas as pd

INPUT_DIR = "/mnt/databases/mri/adni/preproc/4-mni-hist-matching"
REGIONS_DIR = "/mnt/databases/mri/adni/preproc/5-parcellation/regions"
MNI_PATH = "/mnt/study-data/pgirardi/datasets_img/atlases/templates/mni152_2009c_template.nii.gz"
SUFFIX = "_stripped_nlm_denoised_biascorrected_mni_template.nii.gz"
OUT_DIR = "test_dvf"
SEED = 42

DIAGS = ["CN","MCI","AD"]  # ex.: ["AD", "CN", "MCI"]
N_POR_DIAG = 30  # imagens por diagnóstico (1 por paciente)

# robustez vem da métrica/iterações, não do tipo: = _register do 3.1 --src oasis (SyNRA)
SYN_ROBUSTO = {"syn_metric": "CC", "syn_sampling": 4, "reg_iterations": (100, 70, 50, 20)}
# variante → (usar máscara da fixa, parâmetros do SyN); None = padrão ANTsPy (Mattes, (40, 20, 0))
VARIANTES = {
    "sem_mask": (False, None),
    "com_mask": (True, None),
    "syn_robusto": (False, SYN_ROBUSTO),
}
RODAR = ["sem_mask", "com_mask", "syn_robusto"]

# ponytail: só fixa seed; determinismo total exige 1 thread (set_ants_deterministic(True)), SyN fica muito lento
ants.config.set_ants_deterministic(False, SEED)
moving = ants.image_read(MNI_PATH)
moving_mask = ants.get_mask(moving)


def t1_path(img_id):
    return os.path.join(INPUT_DIR, img_id + SUFFIX)


def regions_path(img_id):
    return os.path.join(REGIONS_DIR, img_id + "_regions.nii.gz")


df = pd.read_csv("csvs/adnimerged.csv")
df = df[df["DIAG"].isin(DIAGS)]
df = df[df["ID_IMG"].map(lambda i: os.path.isfile(t1_path(i)) and os.path.isfile(regions_path(i)))]
# 1 imagem sorteada por paciente; paciente não se repete entre diagnósticos (ex.: CN que converteu para MCI)
df = df.sample(frac=1, random_state=SEED).drop_duplicates("ID_PT")
sel = pd.concat([df[df["DIAG"] == d].head(N_POR_DIAG) for d in DIAGS])
for d in DIAGS:
    n = (sel["DIAG"] == d).sum()
    if n < N_POR_DIAG:
        print(f"[WARN] {d}: só {n} imagens disponíveis")
print(sel[["ID_IMG", "ID_PT", "DIAG", "SEX", "AGE", "VISIT"]].to_string(index=False))


def pasta(row):
    return os.path.join(OUT_DIR, f"{row.DIAG}_{row.ID_IMG}")


def registrar(fixed, usar_mask, syn_kw):
    """Rigid -> Affine -> SyN (ordem do prealign_to_target do 2.2). Retorna o resultado do SyN."""
    mk = {"mask": ants.get_mask(fixed)} if usar_mask else {}
    rig = ants.registration(fixed, moving, "Rigid", moving_mask=moving_mask, **mk)
    w = rig["warpedmovout"]
    aff = ants.registration(fixed, w, "Affine", moving_mask=ants.get_mask(w), **mk)
    w = aff["warpedmovout"]
    # SyNOnly: só o deformável ("SyN" refaria o afim sobre a imagem já alinhada)
    return ants.registration(fixed, w, "SyNOnly", moving_mask=ants.get_mask(w), **mk, **(syn_kw or {}))

 ID_IMG      ID_PT DIAG SEX  AGE VISIT
 I34007 031_S_0618   CN   M   76   m06
 I23492 141_S_0767   CN   F   73    sc
I294432 041_S_4427   CN   M   72   v04
I123539 067_S_0257   CN   F   81   m24
 I37848 116_S_1232   CN   F   72    sc
I322057 129_S_4369   CN   M   69   v05
I502018 002_S_5178   CN   M   71   v21
I125201 098_S_0896   CN   M   79   m24
I324988 023_S_4448   CN   F   64   v05
 I80850 003_S_0981   CN   F   85   m12
I241008 024_S_4084   CN   F   68   v02
I225667 116_S_4010   CN   F   71   v02
I287487 002_S_1280   CN   F   76   v06
 I95117 136_S_0186   CN   F   83   m24
I436264 128_S_1242   CN   F   72   m12
 I40549 023_S_1306   CN   F   75    sc
I436705 130_S_0886   CN   F   75   m36
 I86729 023_S_1190   CN   F   78   m12
I100034 941_S_1194   CN   M   86   m12
I193526 011_S_0021   CN   F   75   m24
I229148 031_S_4021   CN   M   67   v02
I403713 011_S_0023   CN   M   80   v21
 I77837 033_S_0923   CN   M   84   m12
I243002 099_S_4104   CN   M   72   v02
I153191 033_S_0741   CN  

## 1. Registro

In [3]:
# for k, row in enumerate(sel.itertuples(), 1):
#     d = pasta(row)
#     os.makedirs(d, exist_ok=True)
#     fixed = ants.image_read(t1_path(row.ID_IMG))
#     if not os.path.isfile(os.path.join(d, "fixed.nii.gz")):
#         ants.image_write(fixed, os.path.join(d, "fixed.nii.gz"))
#         shutil.copy2(regions_path(row.ID_IMG), os.path.join(d, "regions.nii.gz"))
#     for v in RODAR:
#         out = os.path.join(d, f"mni_warped_{v}.nii.gz")
#         if os.path.isfile(out):
#             print(f"[{k}/{len(sel)}] [SKIP] {d} {v}")
#             continue
#         syn = registrar(fixed, *VARIANTES[v])
#         shutil.copy2(syn["fwdtransforms"][0], os.path.join(d, f"warp_{v}_1Warp.nii.gz"))
#         ants.image_write(syn["warpedmovout"], out)  # por último: marca de concluído
#         print(f"[{k}/{len(sel)}] [OK] {d} {v}", flush=True)

## 2. Métricas

In [6]:
def resumo(d, v):
    fixed = ants.image_read(os.path.join(d, "fixed.nii.gz"))
    regions = ants.image_read(os.path.join(d, "regions.nii.gz"))
    w = ants.image_read(os.path.join(d, f"mni_warped_{v}.nii.gz"))
    jac = ants.create_jacobian_determinant_image(fixed, os.path.join(d, f"warp_{v}_1Warp.nii.gz")).numpy()
    m = ants.get_mask(fixed).numpy() > 0
    # hipocampo L/R (17/53): núcleo p/ Jacobiano (d2 inclui corno temporal, que na AD puxa J < 1);
    # dilatado 2 voxels (≈ hippocampus_d2 do 3.2) p/ correlação local
    hippo = ants.threshold_image(regions, 17, 17) + ants.threshold_image(regions, 53, 53)
    core, h = hippo.numpy() > 0, ants.iMath(hippo, "MD", 2).numpy() > 0
    wm = ants.get_mask(w).numpy() > 0
    f, x = fixed.numpy(), w.numpy()
    return {
        "ncc_cerebro": np.corrcoef(f[m], x[m])[0, 1],
        "ncc_hipocampo": np.corrcoef(f[h], x[h])[0, 1],
        "dice_cerebro": 2 * (wm & m).sum() / (wm.sum() + m.sum()),
        "jac_hipocampo_mean": jac[core].mean(),
        "jac_min": jac[m].min(),
        "jac_max": jac[m].max(),
        "jac_neg_%": 100 * (jac[m] <= 0).mean(),
        "jac_std": jac[m].std(),
    }


# ponytail: reaproveita resumo.csv (também gerado por test_dvf/resumo_paralelo.py, ~10 min no server3) e só
# calcula combinações novas; se um warp for refeito, apague a linha dele (ou o CSV) para recalcular
RESUMO_CSV = os.path.join(OUT_DIR, "resumo.csv")
antigo = pd.read_csv(RESUMO_CSV) if os.path.isfile(RESUMO_CSV) else pd.DataFrame(columns=["ID_IMG", "DIAG", "variante"])
feitos = set(zip(antigo["ID_IMG"], antigo["variante"]))
novos = [
    {"ID_IMG": r.ID_IMG, "DIAG": r.DIAG, "variante": v, **resumo(pasta(r), v)}
    for r in sel.itertuples() for v in RODAR
    if (r.ID_IMG, v) not in feitos and os.path.isfile(os.path.join(pasta(r), f"mni_warped_{v}.nii.gz"))
]
print(f"{len(feitos)} combinações do resumo.csv, {len(novos)} calculadas agora")
chaves = pd.DataFrame([(r.ID_IMG, r.DIAG, v) for r in sel.itertuples() for v in RODAR],
                      columns=["ID_IMG", "DIAG", "variante"])
tab = chaves.merge(pd.concat([antigo, pd.DataFrame(novos)], ignore_index=True), on=["ID_IMG", "DIAG", "variante"])
tab.to_csv(RESUMO_CSV, index=False)
print(tab.round(4).to_string(index=False))
print("\nMédia por diagnóstico × variante:")
print(tab.groupby(["DIAG", "variante"]).mean(numeric_only=True).round(4).to_string())

270 combinações do resumo.csv, 0 calculadas agora
 ID_IMG DIAG    variante  ncc_cerebro  ncc_hipocampo  dice_cerebro  jac_hipocampo_mean  jac_min  jac_max  jac_neg_%  jac_std
 I34007   CN    sem_mask       0.6661         0.6377        0.9486              1.0107   0.3378   3.6293        0.0   0.2969
 I34007   CN    com_mask       0.6682         0.7079        0.9108              0.8253   0.2160   2.5850        0.0   0.2779
 I34007   CN syn_robusto       0.7920         0.9229        0.9729              1.0166   0.1322   5.7391        0.0   0.4528
 I23492   CN    sem_mask       0.7979         0.8571        0.9860              0.8935   0.2283   2.3178        0.0   0.2203
 I23492   CN    com_mask       0.7992         0.8563        0.9860              0.9020   0.2305   2.3458        0.0   0.2224
 I23492   CN syn_robusto       0.8763         0.9116        0.9774              0.9406   0.2197   3.5201        0.0   0.2661
I294432   CN    sem_mask       0.8186         0.7150        0.9905         

## 3. Média ± DP por diagnóstico e abordagem

1. Média ± DP (amostral) de cada métrica por diagnóstico × abordagem.
2. Diferença pareada contra `sem_mask` (mesmas imagens): média ± DP, % de imagens com Δ > 0 e p do Wilcoxon. "% Δ > 0" só indica o sentido: correlação e Dice, maior é melhor; Jacobiano não tem "melhor".
3. Kruskal–Wallis entre diagnósticos, por abordagem: o Jacobiano e a correlação no hipocampo separam CN, MCI e AD?

São muitos valores de p (2 abordagens × 3 diagnósticos × 8 métricas): tratar como exploratórios ou corrigir (Holm). Salva `test_dvf/resumo_media_dp.csv` e `test_dvf/resumo_pareado.csv`.

In [7]:
from scipy.stats import kruskal, wilcoxon

METRICAS = ["ncc_cerebro", "ncc_hipocampo", "dice_cerebro", "jac_hipocampo_mean",
            "jac_min", "jac_max", "jac_neg_%", "jac_std"]
ORDEM_DIAG = [d for d in ("CN", "MCI", "AD") if d in set(tab["DIAG"])]
ORDEM_VAR = [v for v in ("sem_mask", "com_mask", "syn_robusto") if v in set(tab["variante"])]
BASE = "sem_mask"


def fmt3(df):
    return df.apply(lambda s: s.map("{:.3f}".format))


# 1) média ± DP por diagnóstico × abordagem (DP amostral, ddof=1)
g = tab.groupby(["DIAG", "variante"])[METRICAS]
media_dp = fmt3(g.mean()) + " ± " + fmt3(g.std())
media_dp.insert(0, "n", g.size())
media_dp = media_dp.reindex(pd.MultiIndex.from_product([ORDEM_DIAG, ORDEM_VAR], names=["DIAG", "abordagem"]))
media_dp.to_csv(os.path.join(OUT_DIR, "resumo_media_dp.csv"))
print("Média ± DP por diagnóstico e abordagem")
display(media_dp)

# 2) diferença pareada (mesmas imagens) contra sem_mask
w = tab.pivot_table(index=["DIAG", "ID_IMG"], columns="variante", values=METRICAS)
linhas = []
for v in (x for x in ORDEM_VAR if x != BASE):
    for d in ORDEM_DIAG:
        for c in METRICAS:
            par = w.loc[d, [(c, BASE), (c, v)]].dropna()
            delta = par[(c, v)] - par[(c, BASE)]
            p = wilcoxon(delta).pvalue if len(delta) and (delta != 0).any() else 1.0
            linhas.append({
                "DIAG": d, "abordagem": f"{v} − {BASE}", "metrica": c, "n": len(delta),
                "delta_media": delta.mean(), "delta_dp": delta.std(),
                "pct_delta_pos": 100 * (delta > 0).mean(), "p_wilcoxon": p,
            })
pareado = pd.DataFrame(linhas)
pareado.to_csv(os.path.join(OUT_DIR, "resumo_pareado.csv"), index=False)
pareado["txt"] = (pareado["delta_media"].map("{:+.3f}".format) + " ± " + pareado["delta_dp"].map("{:.3f}".format)
                  + " (" + pareado["pct_delta_pos"].map("{:.0f}".format) + "% >0; p="
                  + pareado["p_wilcoxon"].map("{:.1e}".format) + ")")
print(f"\nDiferença pareada contra {BASE}: média ± DP (% imagens com Δ > 0; p Wilcoxon)")
display(pareado.pivot_table(index=["abordagem", "DIAG"], columns="metrica", values="txt", aggfunc="first")
        .reindex(index=pd.MultiIndex.from_product([pareado["abordagem"].unique(), ORDEM_DIAG]), columns=METRICAS))

# 3) diagnósticos se separam? (Kruskal–Wallis por abordagem)
print("\nKruskal–Wallis entre diagnósticos (p):")
for v in ORDEM_VAR:
    sub = tab[tab["variante"] == v]
    ps = {c: kruskal(*[sub.loc[sub["DIAG"] == d, c] for d in ORDEM_DIAG]).pvalue
          for c in ("jac_hipocampo_mean", "ncc_hipocampo")}
    print(f"  {v:12s} " + "  ".join(f"{c}: {p:.1e}" for c, p in ps.items()))

Média ± DP por diagnóstico e abordagem


n    ncc_cerebro  ncc_hipocampo   dice_cerebro  \
DIAG abordagem                                                      
CN   sem_mask     30  0.740 ± 0.085  0.708 ± 0.165  0.970 ± 0.023   
     com_mask     30  0.743 ± 0.101  0.714 ± 0.169  0.957 ± 0.044   
     syn_robusto  30  0.817 ± 0.066  0.865 ± 0.073  0.976 ± 0.012   
MCI  sem_mask     30  0.733 ± 0.068  0.707 ± 0.118  0.974 ± 0.018   
     com_mask     30  0.726 ± 0.079  0.714 ± 0.107  0.954 ± 0.048   
     syn_robusto  30  0.810 ± 0.052  0.836 ± 0.075  0.976 ± 0.016   
AD   sem_mask     30  0.661 ± 0.136  0.524 ± 0.200  0.969 ± 0.031   
     com_mask     30  0.667 ± 0.132  0.539 ± 0.192  0.954 ± 0.045   
     syn_robusto  30  0.764 ± 0.111  0.727 ± 0.137  0.975 ± 0.016   

                 jac_hipocampo_mean        jac_min        jac_max  \
DIAG abordagem                                                      
CN   sem_mask         0.902 ± 0.072  0.282 ± 0.077  3.678 ± 0.882   
     com_mask         0.876 ± 0.071  0.250 ± 0.070  3.475 ± 0.998   
     syn_robusto      0.964 ± 0.132  0.176 ± 0.034  4.676 ± 0.866   
MCI  sem_mask         0.886 ± 0.104  0.254 ± 0.081  3.720 ± 0.963   
     com_mask         0.890 ± 0.138  0.246 ± 0.085  3.751 ± 1.314   
     syn_robusto      0.948 ± 0.100  0.172 ± 0.031  4.339 ± 0.859   
AD   sem_mask         0.834 ± 0.092  0.204 ± 0.066  3.635 ± 0.648   
     com_mask         0.844 ± 0.082  0.203 ± 0.076  3.563 ± 0.795   
     syn_robusto      0.888 ± 0.198  0.138 ± 0.040  5.319 ± 2.200   

                      jac_neg_%        jac_std  
DIAG abordagem                                  
CN   sem_mask     0.000 ± 0.000  0.281 ± 0.042  
     com_mask     0.000 ± 0.000  0.277 ± 0.042  
     syn_robusto  0.000 ± 0.000  0.341 ± 0.068  
MCI  sem_mask     0.000 ± 0.000  0.293 ± 0.052  
     com_mask     0.000 ± 0.000  0.286 ± 0.046  
     syn_robusto  0.000 ± 0.000  0.330 ± 0.062  
AD   sem_mask     0.000 ± 0.000  0.298 ± 0.030  
     com_mask     0.000 ± 0.000  0.291 ± 0.031  
     syn_robusto  0.000 ± 0.000  0.384 ± 0.095


Diferença pareada contra sem_mask: média ± DP (% imagens com Δ > 0; p Wilcoxon)


metrica                                             ncc_cerebro  \
com_mask − sem_mask    CN    +0.003 ± 0.065 (63% >0; p=6.6e-01)   
                       MCI   -0.008 ± 0.059 (60% >0; p=8.7e-01)   
                       AD    +0.007 ± 0.063 (67% >0; p=1.6e-01)   
syn_robusto − sem_mask CN    +0.077 ± 0.048 (93% >0; p=1.0e-07)   
                       MCI  +0.077 ± 0.045 (100% >0; p=1.9e-09)   
                       AD    +0.103 ± 0.082 (93% >0; p=1.0e-07)   

metrica                                           ncc_hipocampo  \
com_mask − sem_mask    CN    +0.006 ± 0.092 (73% >0; p=2.8e-02)   
                       MCI   +0.007 ± 0.073 (67% >0; p=9.2e-02)   
                       AD    +0.016 ± 0.076 (70% >0; p=9.6e-02)   
syn_robusto − sem_mask CN   +0.157 ± 0.127 (100% >0; p=1.9e-09)   
                       MCI  +0.128 ± 0.074 (100% >0; p=1.9e-09)   
                       AD   +0.203 ± 0.117 (100% >0; p=1.9e-09)   

metrica                                           dice_cerebro  \
com_mask − sem_mask    CN   -0.013 ± 0.032 (27% >0; p=1.7e-02)   
                       MCI  -0.021 ± 0.042 (40% >0; p=1.9e-02)   
                       AD   -0.015 ± 0.022 (27% >0; p=1.1e-03)   
syn_robusto − sem_mask CN   +0.006 ± 0.020 (50% >0; p=2.6e-01)   
                       MCI  +0.001 ± 0.012 (40% >0; p=9.0e-01)   
                       AD   +0.006 ± 0.021 (40% >0; p=5.7e-01)   

metrica                                     jac_hipocampo_mean  \
com_mask − sem_mask    CN   -0.027 ± 0.073 (47% >0; p=1.5e-01)   
                       MCI  +0.004 ± 0.125 (43% >0; p=4.6e-01)   
                       AD   +0.010 ± 0.073 (47% >0; p=8.4e-01)   
syn_robusto − sem_mask CN   +0.062 ± 0.108 (77% >0; p=2.2e-03)   
                       MCI  +0.062 ± 0.087 (83% >0; p=7.3e-04)   
                       AD   +0.054 ± 0.147 (70% >0; p=2.0e-02)   

metrica                                                jac_min  \
com_mask − sem_mask    CN   -0.033 ± 0.055 (23% >0; p=1.0e-03)   
                       MCI  -0.008 ± 0.050 (47% >0; p=3.4e-01)   
                       AD   -0.001 ± 0.051 (47% >0; p=7.0e-01)   
syn_robusto − sem_mask CN    -0.106 ± 0.073 (3% >0; p=5.6e-09)   
                       MCI  -0.082 ± 0.069 (13% >0; p=3.9e-07)   
                       AD   -0.065 ± 0.054 (17% >0; p=5.7e-07)   

metrica                                                jac_max  \
com_mask − sem_mask    CN   -0.202 ± 0.835 (43% >0; p=2.3e-01)   
                       MCI  +0.031 ± 1.318 (57% >0; p=7.5e-01)   
                       AD   -0.072 ± 0.733 (43% >0; p=2.4e-01)   
syn_robusto − sem_mask CN   +0.998 ± 1.066 (80% >0; p=6.3e-05)   
                       MCI  +0.619 ± 1.301 (70% >0; p=1.4e-02)   
                       AD   +1.684 ± 2.319 (80% >0; p=3.9e-05)   

metrica                                             jac_neg_%  \
com_mask − sem_mask    CN   +0.000 ± 0.000 (0% >0; p=1.0e+00)   
                       MCI  +0.000 ± 0.000 (0% >0; p=1.0e+00)   
                       AD   +0.000 ± 0.000 (0% >0; p=1.0e+00)   
syn_robusto − sem_mask CN   +0.000 ± 0.000 (0% >0; p=1.0e+00)   
                       MCI  +0.000 ± 0.000 (0% >0; p=1.0e+00)   
                       AD   +0.000 ± 0.000 (0% >0; p=1.0e+00)   

metrica                                                jac_std  
com_mask − sem_mask    CN   -0.003 ± 0.036 (47% >0; p=5.0e-01)  
                       MCI  -0.007 ± 0.048 (40% >0; p=5.8e-02)  
                       AD   -0.006 ± 0.033 (37% >0; p=2.9e-01)  
syn_robusto − sem_mask CN   +0.060 ± 0.066 (77% >0; p=2.7e-05)  
                       MCI  +0.037 ± 0.061 (80% >0; p=2.4e-03)  
                       AD   +0.087 ± 0.096 (90% >0; p=6.9e-06)


Kruskal–Wallis entre diagnósticos (p):
  sem_mask     jac_hipocampo_mean: 5.1e-03  ncc_hipocampo: 3.4e-06
  com_mask     jac_hipocampo_mean: 2.9e-01  ncc_hipocampo: 1.4e-05
  syn_robusto  jac_hipocampo_mean: 1.4e-01  ncc_hipocampo: 2.0e-07


## Como as métricas são calculadas

Todas são calculadas no espaço da imagem fixa (ADNI), em duas regiões:

- **Cérebro** (`m`): `ants.get_mask` na fixa. Mantém os voxels acima da intensidade média e limpa com morfologia (erosão, maior componente conexo, dilatação, preenchimento de buracos); aproxima o contorno do cérebro.
- **Hipocampo**: rótulos 17 (esquerdo) e 53 (direito) do `regions` da fixa, em duas versões: **núcleo** (só os voxels rotulados) e **d2** (núcleo dilatado 2 voxels, `iMath MD 2`, ≈ `hippocampus_d2` do `3.2`).

### Sobreposição

- **`dice_cerebro`**: sobreposição entre duas máscaras binárias, a do cérebro na fixa (A) e a do MNI registrado (B, também por `ants.get_mask`): Dice = 2\|A ∩ B\| / (\|A\| + \|B\|), de 0 (nenhuma sobreposição) a 1 (idênticas). Mede só se o **contorno** do cérebro bate, não o alinhamento interno: fica alto mesmo com correspondência interna ruim (I91497: Dice 0,88 com correlação no cérebro 0,30). Serve para detectar falha grosseira.

### Correspondência de intensidade

- **`ncc_cerebro`**: correlação de Pearson entre as intensidades da fixa e do MNI registrado, voxel a voxel, dentro do cérebro (`np.corrcoef(f[m], x[m])`). Registro bom → estruturas nos mesmos voxels → correlação maior.
- **`ncc_hipocampo`**: a mesma conta no hipocampo d2 (alinhamento local). Usa o d2 porque a borda hipocampo/líquor mostra se o contorno foi alinhado.

O nome "ncc" é aproximado: é uma correlação única para a região inteira, diferente da métrica CC que o SyN robusto otimiza (correlação em janelas de raio 4 ao redor de cada voxel).

### Jacobiano

`ants.create_jacobian_determinant_image` calcula, em cada voxel, o determinante da matriz de derivadas da deformação do SyN (`warp_{variante}_1Warp`). Rígido e afim ficam de fora, então J mede a deformação **local**, já descontada a diferença global de tamanho e posição.

- J = 1: volume não muda.
- J < 1: a região correspondente no MNI é menor, ou seja, a estrutura é maior na fixa (ex.: ventrículos aumentados).
- J > 1: a estrutura é menor na fixa que no MNI (o esperado para um hipocampo atrofiado; na prática a AD fica abaixo de 1, ver a justificativa do SyN robusto).
- J ≤ 0: a deformação dobrou o espaço (fisicamente impossível; erro de registro).

Métricas tiradas do mapa de J:

- **`jac_hipocampo_mean`**: média de J no **núcleo** do hipocampo (o d2 inclui o corno temporal, que aumenta na AD e puxa J para baixo).
- **`jac_min`** / **`jac_max`**: menor e maior J no cérebro (compressão e expansão mais extremas).
- **`jac_neg_%`**: % de voxels do cérebro com J ≤ 0 (controle de dobras).
- **`jac_std`**: DP de J no cérebro (quanto a deformação varia no espaço; SyN mais agressivo → DP maior).

Leitura: correlações e Dice dizem se o registro está bom; as métricas de J dizem se a deformação é válida e o que ela mostra sobre a anatomia.

## Como as estatísticas são calculadas

Unidade de análise: a imagem (1 por paciente, 30 por diagnóstico). Todas as variantes rodaram nas mesmas 90 imagens, então a comparação entre variantes é **pareada** (cada imagem é comparada com ela mesma), e a comparação entre diagnósticos é entre grupos **independentes** (imagens diferentes).

### Descritivas

- **Média ± DP**: média e desvio padrão amostral (`ddof=1`, divide por n − 1) de cada métrica, por diagnóstico × variante (`test_dvf/resumo_media_dp.csv`).
- **Mediana**: valor do meio, depois de ordenar. Usada no texto para o J do hipocampo, porque não é puxada por casos extremos (ex.: I107707, J = 0,29).

### Comparação entre variantes (pareada)

- **Δ**: para cada imagem, Δ = métrica na variante − métrica no `sem_mask`. Reporta média ± DP de Δ e **% de imagens com Δ > 0** (para correlações e Dice, Δ > 0 = melhorou; para J, só indica o sentido da mudança) (`test_dvf/resumo_pareado.csv`).
- **Teste de Wilcoxon de postos sinalizados** (`scipy.stats.wilcoxon(delta)`, bilateral): testa se os Δ estão centrados em zero, ou seja, se a variante muda a métrica de forma consistente. Ordena os \|Δ\| do menor para o maior, dá a cada um seu posto (1, 2, 3…) e soma separadamente os postos dos Δ positivos e dos negativos; se a variante não mudasse nada, as duas somas seriam parecidas. Não supõe distribuição normal (os Δ têm caudas longas, ex.: −0,342 a +0,252 na correlação do hipocampo). Δ = 0 são descartados; se todos forem 0 (ex.: `jac_neg_%`), p = 1. Com n = 30 o p é exato: se os 30 Δ têm o mesmo sinal, p = 2/2³⁰ ≈ 2e-9, o menor possível (por isso esse valor se repete na tabela do SyN robusto). Na justificativa da máscara, o mesmo teste também é aplicado às 90 imagens juntas.

### Comparação entre diagnósticos (independente)

- **Teste de Kruskal–Wallis** (`scipy.stats.kruskal`), dentro de cada variante: testa se CN, MCI e AD vêm da mesma distribuição. Junta as 90 imagens, ordena e dá postos, e compara o posto médio de cada grupo; a estatística H segue aproximadamente χ² com 2 graus de liberdade. p pequeno = pelo menos um grupo difere dos outros (não diz qual). Também não supõe normalidade.

### Associação

- **Correlação de Pearson r** (justificativa do SyN robusto): entre o valor de uma métrica no padrão e o ganho do robusto na mesma imagem (de −1 a 1; r < 0 = quem estava pior ganha mais). Parte de um r negativo é esperada por construção, porque o valor no padrão entra nas duas variáveis (ganho = robusto − padrão) e há teto perto de 1.

### Como ler os p

- Referência p < 0,05, **sem correção para comparações múltiplas** (8 métricas × 3 diagnósticos × 2 comparações = 48 testes na tabela pareada; ~2 dariam p < 0,05 por acaso). Um p entre 0,01 e 0,05 isolado é evidência fraca; padrões que se repetem nos três diagnósticos pesam mais.
- p mede consistência, não tamanho. O tamanho do efeito é comparado com a **variação entre execuções** (~0,02 na correlação, estimada repetindo o registro de I7055 sem máscara, porque o ANTs com várias threads não é determinístico): Δ menor que isso não é relevante na prática, mesmo com p pequeno.

## Justificativa e resultados do teste de máscara (30 CN, 30 MCI, 30 AD)

**Objetivo.** Verificar a afirmação da auditoria de que `fixed_mask`, usado em `2.2_groupwise_ants.py`, é ignorado pelo ANTsPy, e medir quanto a máscara da imagem fixa (`mask`) altera o corregistro.

**Evidência no código (antspyx 0.6.3).**

- A assinatura de `ants.registration` tem `mask` (fixa), `moving_mask` e `mask_all_stages`, mas não tem `fixed_mask`. O argumento cai em `**kwargs`, que não é usado: é descartado sem erro nem aviso.
- As máscaras viram a opção `-x [fixa,móvel]` do `antsRegistration`; sem `mask`, a parte fixa é `NA`. Em transformações com vários estágios (`SyN`, `SyNRA`), sem `mask_all_stages=True` a máscara só vale no último estágio.
- Portanto, a variante `sem_mask` reproduz exatamente o que o 2.2 executou: só `moving_mask`.

**Desenho.** MNI152 2009c (móvel) → imagem ADNI (fixa). Rigid → Affine → SyNOnly, na mesma ordem de `prealign_to_target` do 2.2. A única diferença entre `sem_mask` e `com_mask` é a máscara da imagem fixa (`ants.get_mask`). As duas variantes rodaram nas mesmas 90 imagens (30 CN, 30 MCI, 30 AD). Comparação pareada (`test_dvf/resumo.csv`; Wilcoxon).

**Resultados (90 imagens, média ± DP).**

| Métrica | sem `mask` | com `mask` | Δ (com − sem) | % imagens com Δ > 0 | p |
|---|---|---|---|---|---|
| Correlação de intensidade no cérebro | 0,711 ± 0,106 | 0,712 ± 0,110 | +0,001 ± 0,062 | 63% | 0,26 |
| Correlação no hipocampo (d2) | 0,646 ± 0,184 | 0,656 ± 0,179 | +0,010 ± 0,080 | 70% | 0,002 |
| Dice do cérebro | 0,971 ± 0,025 | 0,955 ± 0,045 | −0,016 ± 0,033 | 31% | 3e-6 |
| J no hipocampo (núcleo) | 0,874 ± 0,094 | 0,870 ± 0,102 | −0,004 ± 0,094 | 46% | 0,21 |
| J mín. / máx. | 0,247 / 3,68 | 0,233 / 3,60 | −0,014 / −0,081 | 39% / 48% | 0,006 / 0,10 |
| DP do Jacobiano | 0,290 | 0,285 | −0,006 | 41% | 0,036 |
| Voxels com J ≤ 0 | 0% | 0% | 0 | — | — |

Por diagnóstico, o Dice cai nos três grupos (CN −0,013, p = 0,017; MCI −0,021, p = 0,019; AD −0,015, p = 0,001). O ganho na correlação do hipocampo é parecido nos três (CN +0,006, p = 0,028; MCI +0,007, p = 0,09; AD +0,016, p = 0,10).

**Interpretação.**

- **O efeito mais consistente da máscara é piorar o Dice do cérebro** (−0,016; p = 3e-6; igual em CN, MCI e AD). Só 17 das 90 imagens melhoram mais que 0,001; as maiores quedas são I273330 (MCI, −0,193), I659919 (CN, −0,110) e I123539 (CN, −0,105). O Dice mínimo cai de 0,88 para 0,79. Explicação provável: as imagens já estão sem crânio, com fundo zero; sem máscara, a borda cérebro/fundo entra na métrica e ajuda a alinhar o contorno, e a máscara corta essa borda.
- A correlação no hipocampo sobe em 70% das imagens, mas o ganho médio (+0,010) é metade da variação entre execuções estimada antes numa repetição de I7055 sem máscara (~0,02; ANTs com várias threads não é determinístico), e é instável por imagem (de −0,342 em I659919 a +0,252 em I324988). Para comparar: o SyN robusto melhora a correlação no hipocampo em +0,153 sobre o `com_mask`, nas 90 imagens (p = 2e-16).
- Correlação no cérebro e J médio no hipocampo não mudam (p ≥ 0,21). Com máscara, o J mínimo fica um pouco menor (mais compressão local; p = 0,006).
- **A máscara reduz a separação entre diagnósticos no J do hipocampo:** Kruskal–Wallis p = 0,005 sem máscara contra 0,29 com máscara (na correlação do hipocampo, os dois separam: p = 3e-6 e 1e-5).
- A máscara muda o registro já nos estágios lineares (em I7055: parâmetros do rígido até 1,87 e do afim até 3,61; deslocamento do SyN com diferença média de 1,85 mm); parte da diferença no SyN é compensação dos alinhamentos lineares diferentes, não diferença anatômica.
- A configuração sem máscara (a que o 2.2 executou) gerou deformações válidas nas 90 imagens: nenhum voxel com J ≤ 0 e Dice do cérebro 0,88–0,99. A correspondência, porém, é fraca em algumas AD (correlação no cérebro 0,30–0,49 em I91497, I174215, I117597 e I13003; ver o teste do SyN robusto).

**Conclusões.**

1. O bug é real, mas só afeta o `2.2_groupwise_ants.py` (templates ADNI, usados nas referências `disp_ad` e `disp_cnad`). Esses templates foram construídos, na prática, sem máscara fixa, e o teste indica que isso não é prejudicial: a máscara piora o contorno (Dice) nos três diagnósticos e dá um ganho no hipocampo menor que a variação entre execuções. Correção recomendada: remover `fixed_mask` e documentar "sem máscara fixa", em vez de trocar por `mask`.
2. O pipeline OASIS não passa máscara em nenhum registro: o rígido do `2.3` usa `corregistro_rigid_mni` sem máscara, o `ants.build_template` recebe só `TEMPLATE_REG`, e o `SyNRA` do `3.1 --src oasis` é chamado sem `mask`/`moving_mask` (equivale a `-x [NA,NA]`). O que foi executado é o que estava especificado; o bug não se aplica.
3. Limitações: configuração diferente da do OASIS (MNI como móvel, métrica Mattes, `SyNOnly` 40x20x0); variação entre execuções estimada com uma única repetição de uma imagem.

## Justificativa e resultados do SyN robusto (parâmetros do `3.1 --src oasis`), 30 CN, 30 MCI e 30 AD

Mesmo rígido → afim, sem máscara fixa. Muda só o estágio deformável:

- `sem_mask` (padrão): Mattes, `reg_iterations=(40, 20, 0)`, sem iterações na resolução máxima.
- `syn_robusto`: CC raio 4, `reg_iterations=(100, 70, 50, 20)`, igual ao `SyNRA` do pipeline OASIS. O tipo continua `SyNOnly`: no ANTsPy o tipo só escolhe os estágios lineares; a robustez vem da métrica e das iterações.

Critério de melhora: `ncc_hipocampo` e `ncc_cerebro` maiores, sem Jacobiano ≤ 0. `jac_hipocampo_mean` é calculado no núcleo do hipocampo (sem dilatação); J < 1 = estrutura maior na imagem fixa que no MNI.

**Sujeitos.** 30 por diagnóstico, sorteados com `SEED = 42`: 1 imagem por paciente (qualquer visita), paciente não se repete entre diagnósticos (`test_dvf/selected_images.csv`). Idade média: CN 75,0 (64–86), MCI 72,2 (57–88), AD 76,2 (61–89).

**Resultados** (média ± DP por grupo; `test_dvf/resumo.csv`, gerado por `test_dvf/resumo_paralelo.py`).

| Métrica | CN padrão | CN robusto | MCI padrão | MCI robusto | AD padrão | AD robusto |
|---|---|---|---|---|---|---|
| Correlação no cérebro | 0,740 ± 0,085 | 0,817 ± 0,066 | 0,733 ± 0,068 | 0,810 ± 0,052 | 0,661 ± 0,136 | 0,764 ± 0,111 |
| Correlação no hipocampo (d2) | 0,708 ± 0,165 | 0,865 ± 0,073 | 0,707 ± 0,118 | 0,836 ± 0,075 | 0,524 ± 0,200 | 0,727 ± 0,137 |
| Dice do cérebro | 0,970 ± 0,023 | 0,976 ± 0,012 | 0,974 ± 0,018 | 0,976 ± 0,016 | 0,969 ± 0,031 | 0,975 ± 0,016 |
| J no hipocampo (núcleo) | 0,902 ± 0,072 | 0,964 ± 0,132 | 0,886 ± 0,104 | 0,948 ± 0,100 | 0,834 ± 0,092 | 0,888 ± 0,198 |
| J mín. / máx. (média) | 0,28 / 3,68 | 0,18 / 4,68 | 0,25 / 3,72 | 0,17 / 4,34 | 0,20 / 3,64 | 0,14 / 5,32 |
| DP do Jacobiano | 0,281 | 0,341 | 0,293 | 0,330 | 0,298 | 0,384 |
| Voxels com J ≤ 0 | 0% | 0% | 0% | 0% | 0% | 0% |

Diferença pareada robusto − padrão (mesmas imagens; % de imagens com Δ > 0; Wilcoxon):

| Métrica | CN | MCI | AD |
|---|---|---|---|
| Correlação no cérebro | +0,077 (93%; p = 1e-7) | +0,077 (100%; p = 2e-9) | +0,103 (93%; p = 1e-7) |
| Correlação no hipocampo | +0,157 (100%; p = 2e-9) | +0,128 (100%; p = 2e-9) | +0,203 (100%; p = 2e-9) |
| Dice do cérebro | +0,006 (50%; p = 0,26) | +0,001 (40%; p = 0,90) | +0,006 (40%; p = 0,57) |
| J no hipocampo | +0,062 (77%; p = 0,002) | +0,062 (83%; p = 7e-4) | +0,054 (70%; p = 0,02) |

**Interpretação.**

- **A correspondência no hipocampo melhora em todas as 90 imagens** (+0,13 a +0,20 em média), e a do cérebro em 86 das 90. O ganho é maior na AD (+0,203 no hipocampo), onde o padrão era pior, e quanto pior a imagem no padrão, maior o ganho (correlação de Pearson entre valor no padrão e ganho no hipocampo: r = −0,80). Os ganhos médios são 4 a 10 vezes a variação entre execuções estimada no teste de máscara (~0,02).
- **A qualidade do registro depende do diagnóstico nas duas variantes:** a correlação no hipocampo difere entre CN, MCI e AD (Kruskal–Wallis p = 3e-6 no padrão, 2e-7 no robusto), com a AD pior (0,73 contra 0,87 no CN no robusto). Com atrofia acentuada, a correspondência local com um template de adultos jovens segue imperfeita. Consequência para os atributos de deformação: parte da diferença entre grupos pode ser erro de registro correlacionado com o diagnóstico, e não só anatomia.
- **Dice do cérebro não muda** (Δ ≤ 0,006, p ≥ 0,26) e fica ≥ 0,91 em todas as imagens no robusto (≥ 0,88 no padrão). A queda leve vista nas 6 imagens anteriores não se confirmou.
- **Deformações mais intensas, mas válidas:** J mínimo cai, J máximo e DP do Jacobiano sobem nos três grupos, e nenhum voxel tem J ≤ 0 nas 180 deformações. Extremo: I348196 (AD, Philips 3T) com J máx. 14,1 no robusto (o 2º maior é 8,3), com boa correspondência no hipocampo (0,79); é local, mas vale QC visual.
- **Imagens com registro ruim no padrão melhoram, mas algumas seguem ruins:** I13003 (AD; correlação no cérebro 0,49 → 0,79) e I174215 (AD; 0,37 → 0,69) se recuperam; I91497 (AD; 0,30 → 0,49, hipocampo −0,11 → 0,21), I117597 (AD; 0,41 → 0,36, hipocampo 0,10 → 0,41) e I123539 (CN; 0,49 → 0,59) continuam abaixo de 0,6. Conferir no ITK-SNAP se o problema é de pré-processamento (crop, skull stripping, hist-match) antes de usar essas imagens.
- **J no hipocampo mantém a ordem esperada, mas separa menos os grupos no robusto.** Medianas CN > MCI > AD nas duas variantes (padrão 0,905 / 0,870 / 0,828; robusto 0,948 / 0,933 / 0,878), e o robusto sobe J de forma parecida nos três (+0,05 a +0,06). Mas a dispersão aumenta (DP na AD 0,092 → 0,198; I107707 com J = 0,29) e a diferença entre diagnósticos deixa de ser significativa (Kruskal–Wallis p = 0,005 no padrão, 0,14 no robusto). A AD fica abaixo de 1 nas duas variantes; na rodada anterior de 6 imagens, o J do corno temporal vizinho (0,60–0,91) e dos ventrículos laterais (0,32–0,81, menor na AD) confirmou o sinal do Jacobiano (não recalculado aqui).

**Conclusões.**

1. O SyN robusto (CC raio 4, `100x70x50x20`) registra melhor que o padrão do ANTsPy: melhora a correspondência no hipocampo em todas as 90 imagens, mais na AD e nas imagens piores, sem dobras e sem perder Dice. Isso sustenta a configuração do pipeline OASIS e indica que os registros com SyN padrão (`2.2` e o caminho ADNI do `3.1`, `type_of_transform="SyN"` sem parâmetros) têm correspondência local pior.
2. O Jacobiano médio no hipocampo contra o MNI não é um marcador direto de atrofia: com o registro melhor, a separação entre diagnósticos até diminui. Deve ser lido de forma relativa (por exemplo `logjac_rel_mean` do `3.2`, que desconta a deformação global) e contra templates normativos pareados por idade, como os templates OASIS por década.
3. A qualidade do registro varia com o diagnóstico mesmo no robusto; reportar QC de registro por grupo junto com os resultados dos atributos de deformação.
4. Limitações: MNI como móvel, e não o template OASIS; execução com várias threads (não determinística); variação entre execuções estimada com uma única repetição de uma imagem; correlação de intensidade como medida de correspondência (não há segmentação manual para comparar).

**Tempo observado:** ~15 min por imagem no robusto contra ~1 min no padrão (várias threads, servidor compartilhado).